# Tabular Q-Learning Training Notebook
This notebook trains and evaluates the tabular Q-learning model.

## 1. Imports & Paths

In [1]:
import os
import sys
from pathlib import Path
import torch

# Set up project root
PROJECT_ROOT = Path(os.getcwd()).resolve().parent
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/scottyang/MLB-Bullpen-Strategy


In [2]:
from src.rl.tabular_q_agent import (
    load_tabular_q_config,
    train_tabular_q_agent,
    TabularOfflineDataset,
)

## 2. Configurations

In [3]:
DATA_DIR = PROJECT_ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

YEAR_TAG = "2022_2023"
RL_TENSORS_PATH = PROC_DIR / f"rl_tensors_{YEAR_TAG}_constrained.npz"
MODEL_CFG_PATH = CONFIG_DIR / "model.yaml"
TRAIN_CFG_PATH = CONFIG_DIR / "training.yaml"
MODEL_OUT_PATH = MODELS_DIR / f"constrained_cql_model_{YEAR_TAG}.pt"

print("RL tensors:", RL_TENSORS_PATH)
print("Model config:", MODEL_CFG_PATH)
print("Training config:", TRAIN_CFG_PATH)
print("Model output:", MODEL_OUT_PATH)

RL tensors: /Users/scottyang/MLB-Bullpen-Strategy/data/processed/rl_tensors_2022_2023_constrained.npz
Model config: /Users/scottyang/MLB-Bullpen-Strategy/configs/model.yaml
Training config: /Users/scottyang/MLB-Bullpen-Strategy/configs/training.yaml
Model output: /Users/scottyang/MLB-Bullpen-Strategy/models/constrained_cql_model_2022_2023.pt


## 3. Load Tabular Q-Learning Configuration

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

train_cfg = load_tabular_q_config(
    model_config_path=MODEL_CFG_PATH,
    data_path=RL_TENSORS_PATH,
    device=device,
)

train_cfg

Using device: cpu


TabularQAgentConfig(data_path=PosixPath('/Users/scottyang/MLB-Bullpen-Strategy/data/processed/rl_tensors_2022_2023_constrained.npz'), device='cpu', alpha=0.05, gamma=0.99, num_epochs=25, val_fraction=0.1, precision=4, log_interval=1, yaml_num_actions=11)

## 4. Load Offline Dataset

In [5]:
ds = TabularOfflineDataset(
    data_path=train_cfg.data_path,
    device=train_cfg.device,
)

print("Dataset size:", len(ds))
print("Num actions:", ds.num_actions)
print("Example state vector shape:", ds.state_vec[0].shape)

Dataset size: 22249
Num actions: 11
Example state vector shape: torch.Size([13])


## 5. Train Models and Save

In [6]:
tabular_agent = train_tabular_q_agent(train_cfg)

[Tabular-Q] epoch=0/25  val_td_error=0.475603
[Tabular-Q] epoch=1/25  val_td_error=0.492449
[Tabular-Q] epoch=2/25  val_td_error=0.507144
[Tabular-Q] epoch=3/25  val_td_error=0.520304
[Tabular-Q] epoch=4/25  val_td_error=0.537374
[Tabular-Q] epoch=5/25  val_td_error=0.551955
[Early Stopping] No improvement for 5 epochs. Stopping.


In [7]:
tabular_agent.save(MODEL_OUT_PATH)
MODEL_OUT_PATH

PosixPath('/Users/scottyang/MLB-Bullpen-Strategy/models/constrained_cql_model_2022_2023.pt')

## 6. Offline TD error (Bellman Residual)



In [8]:
import numpy as np

def tabular_td_error(agent, ds, gamma):
    total = 0.0
    n = 0
    for i in range(len(ds)):
        s, a, r, ns, done, mask = ds[i]

        s_key = agent._s(s)
        ns_key = agent._s(ns)

        q_sa = agent.Q[s_key][a]
        target = r if done else r + gamma * np.max(agent.Q[ns_key])

        total += (q_sa - target)**2
        n += 1

    return total / max(n, 1)

mste = tabular_td_error(tabular_agent, ds, train_cfg.gamma)
print("Mean Squared TD Error:", mste)


Mean Squared TD Error: 0.47237572


## 7. Direct Q-Based Estimate of Greedy Policy



In [9]:
def tabular_direct_value(agent, ds):
    values = []
    for i in range(len(ds)):
        s, _, _, _, _, mask = ds[i]

        s_key = agent._s(s)
        q = agent.Q[s_key].copy()

        # Apply availability mask
        q[~mask] = -1e9

        values.append(np.max(q))

    return np.mean(values)

dm_value = tabular_direct_value(tabular_agent, ds)
print("Direct Greedy Policy Value:", dm_value)

Direct Greedy Policy Value: 0.1666087


## 8. Action agreement with logged policy

How often does the greedy Tabular Q-Learning action (respecting availability mask) match the logged (historical) action from the dataset?


In [10]:
def tabular_action_agreement(agent, ds):
    matches = 0
    for i in range(len(ds)):
        s, logged_a, _, _, _, mask = ds[i]
        greedy_a = agent.act(s, mask)
        matches += int(greedy_a == logged_a)

    return matches / len(ds)

agreement = tabular_action_agreement(tabular_agent, ds)
print(f"Action Agreement: {agreement:.2%}")

Action Agreement: 50.40%


## 9. Summary



In [14]:
print("=========== TABULAR CONSTRAINED Q LEARNING RESULTS ===========")
print(f"Mean Squared TD Error:         {mste:.6f}")
print(f"Direct Greedy Policy Value:    {dm_value:.6f}")
print(f"Action Agreement vs MLB:       {agreement:.2%}")

from src.rl.tabular_q_agent import TabularOfflineDataset, compute_pull_rate

ds = TabularOfflineDataset(RL_TENSORS_PATH, device="cpu")
pull_rate = compute_pull_rate(tabular_agent, ds)
print(f"Pull Rate: {pull_rate*100:.2f}%")

=========== TABULAR CONSTRAINED Q LEARNING RESULTS ===========
Mean Squared TD Error:         0.472376
Direct Greedy Policy Value:    0.166609
Action Agreement vs MLB:       50.40%
Pull Rate: 47.41%
